# 🚀 Quick Document Analyzer

**Purpose**: Upload a NEW RFP and automatically generate the proposal approach section.

---

## 📋 How This Works:

1. **Upload or Select RFP** → You provide the RFP to analyze
2. **System Learns** → RAG engine retrieves similar patterns from your repository
3. **Generate Approach** → AI agents build the complete proposal structure
4. **Review & Export** → You get the approach section in Excel/PowerPoint

---

## ⚙️ Prerequisites:

1. ✅ Your `.env` file is configured with OpenAI API key
2. ✅ `data/rfps/` folder has your historical RFPs (for learning)
3. ✅ `data/proposals/` folder has your historical proposals (for learning)

---

## 1️⃣ Setup & Initialize System

In [ ]:
# ============================================
# 🔄 AUTO-UPDATE SCRIPT
# ============================================
# Run this when you hit an error - it fixes everything automatically!
# ============================================

import subprocess
import importlib
import sys
from pathlib import Path

def auto_update():
    print("🔄 AUTO-UPDATE STARTING...\n")
    print("=" * 60)
    
    # Get repository root
    notebook_dir = Path.cwd()
    repo_root = notebook_dir.parent if notebook_dir.name == 'notebooks' else notebook_dir
    print(f"📁 Repository: {repo_root}")
    
    # Step 1: Stash local changes
    print("\n💾 Saving local changes...")
    try:
        result = subprocess.run(["git", "stash"], cwd=str(repo_root), 
                              capture_output=True, text=True, timeout=10)
        print("   ✅ Changes saved" if "Saved" in result.stdout else "   ✅ No changes to save")
    except Exception as e:
        print(f"   ⚠️ {e}")
    
    # Step 2: Pull latest code
    print("\n⬇️ Pulling latest fixes...")
    try:
        result = subprocess.run(
            ["git", "pull", "origin", "claude/rebuild-proposal-builder-011CUXXm3yt3mm1e8cP6UcMb"],
            cwd=str(repo_root), capture_output=True, text=True, timeout=30
        )
        print("   ✅ Up to date" if "Already up to date" in result.stdout else "   ✅ Updates pulled")
    except Exception as e:
        print(f"   ⚠️ {e}")
    
    # Step 3: Reload modules
    print("\n🔥 Reloading modules...")
    for mod in ['data_models', 'rag_engine', 'agents', 'utils', 'visualizations', 'exports']:
        if mod in sys.modules:
            try:
                importlib.reload(sys.modules[mod])
                print(f"   ✅ {mod}")
            except Exception as e:
                print(f"   ⚠️ {mod}: {e}")
    
    # Step 4: Re-import
    print("\n📦 Re-importing...")
    try:
        from data_models import ProjectStructure
        from rag_engine import RAGEngine
        from agents import WorkstreamAgent, DependencyAgent, ModuleAgent, TimelineOptimizer
        from utils import validate_environment, save_project
        from visualizations import ProjectVisualizer
        from exports import export_project
        print("   ✅ All imports successful")
    except Exception as e:
        print(f"   ⚠️ {e}")
    
    # Step 5: Check preserved variables
    print("\n🔍 Your variables:")
    for var in ['rag_engine', 'rfp_text', 'enhanced_workstreams', 'project']:
        if var in globals():
            print(f"   ✅ {var} preserved")
    
    print("\n" + "=" * 60)
    print("🎉 DONE! Re-run the error cell now.")
    print("=" * 60)

auto_update()

## 🔄 Auto-Update (Run This When Errors Occur!)

**Encountered an error?** Just run the cell below to automatically:
- ✅ Pull latest fixes from GitHub
- ✅ Reload all Python modules
- ✅ Preserve your variables (rag_engine, rfp_text, etc.)
- ✅ No manual steps needed!

**Then re-run the cell that had an error.**

In [ ]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv
import time

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

# Load environment
load_dotenv(Path.cwd().parent / '.env')

# Import modules
from data_models import ProjectStructure
from rag_engine import RAGEngine
from agents import WorkstreamAgent, DependencyAgent, ModuleAgent, TimelineOptimizer
from utils import validate_environment, save_project
from visualizations import ProjectVisualizer
from exports import export_project

print("✅ All modules imported successfully!")
print("\n📝 Validating environment...\n")

# Validate environment
validation = validate_environment()

if validation['valid']:
    print("✅ Environment is ready!")
else:
    print("❌ Environment validation failed:")
    for error in validation['errors']:
        print(f"   - {error}")
    print("\n⚠️ Please fix these issues before continuing.")

if validation.get('warnings'):
    print("\n⚠️ Warnings:")
    for warning in validation['warnings']:
        print(f"   - {warning}")

## 2️⃣ Initialize RAG Engine (Smart Indexing)

This indexes your historical RFPs and Proposals so the system can learn from them.

**⏱️ First run: Takes 2-3 minutes to index your documents**  
**🚀 Subsequent runs: Automatically detects existing database and skips indexing!**

💡 **Smart Features:**
- Checks if database already exists
- Skips re-indexing to save time and API costs
- Only indexes once unless you delete the database folder

**You can safely re-run this cell - it won't waste your OpenAI credits!**

In [ ]:
# Initialize RAG Engine
api_key = os.getenv('OPENAI_API_KEY')
db_path = Path.cwd().parent / 'data' / 'chroma_db'
embed_model = os.getenv('EMBED_MODEL', 'text-embedding-3-large')

print("🔧 Initializing RAG Engine...")
rag_engine = RAGEngine(
    api_key=api_key,
    db_path=str(db_path),
    embed_model=embed_model,
    verbose=True
)

print("✅ RAG Engine initialized!\n")

# Check repository status
print("📊 Checking repository...\n")

rfp_dir = Path.cwd().parent / 'data' / 'rfps'
proposal_dir = Path.cwd().parent / 'data' / 'proposals'

rfp_count = len(list(rfp_dir.glob('*.pdf'))) if rfp_dir.exists() else 0
proposal_count = len(list(proposal_dir.glob('*.pptx'))) if proposal_dir.exists() else 0

print(f"📁 Found {rfp_count} RFPs and {proposal_count} Proposals in repository")

if rfp_count == 0 or proposal_count == 0:
    print("\n⚠️ WARNING: Your repository is empty or incomplete!")
    print("   Please add your historical RFPs (PDFs) to data/rfps/")
    print("   Please add your historical Proposals (PPTX) to data/proposals/")
else:
    # Check if database already exists and has content
    db_exists = db_path.exists() and any(db_path.iterdir())
    
    if db_exists:
        print("\n✅ Vector database already exists!")
        print("   Skipping indexing to save time and API costs.")
        print("   Your system is ready to use the existing knowledge base.")
        print("\n💡 To re-index (if you added new files):")
        print("   1. Delete the data/chroma_db/ folder")
        print("   2. Re-run this cell")
    else:
        # Index documents
        print("\n🔄 Indexing documents (this may take 2-3 minutes)...\n")
        print("💡 This only needs to be done ONCE. The database will be saved for future use.")
        
        start_time = time.time()
        result = rag_engine.index_documents(
            rfp_dir=str(rfp_dir),
            proposal_dir=str(proposal_dir)
        )
        duration = time.time() - start_time
        
        print(f"\n✅ Indexing complete!")
        print(f"   - Total chunks: {result['total_chunks']}")
        print(f"   - RFPs indexed: {result['rfp_count']}")
        print(f"   - Proposals indexed: {result['proposal_count']}")
        print(f"   - Duration: {duration:.1f} seconds")
        print(f"\n💾 Database saved to: {db_path}")
        print("   You won't need to re-index next time!")

---

## 3️⃣ Upload or Select NEW RFP to Analyze

### 📤 Option A: Upload a NEW RFP file (from anywhere on your computer)

In [ ]:
from ipywidgets import FileUpload, Button, Output, VBox
from IPython.display import display, clear_output
import pdfplumber

# File upload widget
upload_widget = FileUpload(
    accept='.pdf',
    multiple=False,
    description='Upload RFP (PDF)'
)

output = Output()

rfp_text = None
rfp_filename = None

def on_upload_change(change):
    global rfp_text, rfp_filename

    with output:
        clear_output()

        # Check if file was uploaded
        if not upload_widget.value or len(upload_widget.value) == 0:
            print("⚠️ No file selected")
            return

        try:
            # Handle different value structures (ipywidgets version compatibility)
            uploaded_files = upload_widget.value

            # Get first file - handle both tuple and dict formats
            if isinstance(uploaded_files, dict):
                # Dictionary format: {filename: {metadata}}
                rfp_filename = list(uploaded_files.keys())[0]
                uploaded_file = uploaded_files[rfp_filename]
                file_content = uploaded_file.get('content', uploaded_file.get('data'))
            elif isinstance(uploaded_files, (list, tuple)) and len(uploaded_files) > 0:
                # Tuple/list format: [metadata, ...]
                uploaded_file = uploaded_files[0]
                if isinstance(uploaded_file, dict):
                    rfp_filename = uploaded_file.get('name', 'uploaded.pdf')
                    file_content = uploaded_file.get('content', uploaded_file.get('data'))
                else:
                    # Direct file object
                    rfp_filename = getattr(uploaded_file, 'name', 'uploaded.pdf')
                    file_content = getattr(uploaded_file, 'content', None)
            else:
                print(f"⚠️ Unexpected upload format: {type(uploaded_files)}")
                print(f"   Value: {uploaded_files}")
                return

            if file_content is None:
                print("❌ Could not extract file content")
                return

            print(f"📄 Processing: {rfp_filename}")

            # Save temporarily
            temp_path = Path.cwd().parent / 'data' / 'temp_rfp.pdf'
            temp_path.parent.mkdir(parents=True, exist_ok=True)

            with open(temp_path, 'wb') as f:
                f.write(file_content)

            # Extract text
            print("🔄 Extracting text...")
            text_parts = []

            with pdfplumber.open(temp_path) as pdf:
                for page_num, page in enumerate(pdf.pages, 1):
                    page_text = page.extract_text()
                    if page_text:
                        text_parts.append(f"\n--- Page {page_num} ---\n{page_text}")

            rfp_text = "\n".join(text_parts)

            print(f"✅ Successfully extracted {len(rfp_text)} characters from {len(text_parts)} pages")
            print(f"\n📝 Preview (first 500 chars):\n")
            print(rfp_text[:500] + "...")

            # Clean up
            temp_path.unlink()

        except Exception as e:
            print(f"❌ Error processing file: {e}")
            import traceback
            print("\n🔍 Debug info:")
            print(f"   upload_widget.value type: {type(upload_widget.value)}")
            traceback.print_exc()
            rfp_text = None

upload_widget.observe(on_upload_change, names='value')

display(VBox([upload_widget, output]))

print("\n💡 Click 'Upload' above and select your RFP PDF file")

### 📂 Option B: Select from RFPs already in your repository

In [ ]:
from ipywidgets import Dropdown, Button, Output, VBox
from IPython.display import display

# List available RFPs
rfp_dir = Path.cwd().parent / 'data' / 'rfps'
available_rfps = list(rfp_dir.glob('*.pdf')) if rfp_dir.exists() else []

if not available_rfps:
    print("⚠️ No RFPs found in data/rfps/ folder")
    print("   Please use Option A to upload a new RFP")
else:
    print(f"📁 Found {len(available_rfps)} RFPs in your repository:\n")
    
    # Dropdown widget
    rfp_dropdown = Dropdown(
        options=[(rfp.name, rfp) for rfp in available_rfps],
        description='Select RFP:',
        style={'description_width': 'initial'}
    )
    
    load_button = Button(description='Load Selected RFP', button_style='primary')
    output = Output()
    
    def on_load_click(b):
        global rfp_text, rfp_filename
        
        with output:
            output.clear_output()
            
            selected_path = rfp_dropdown.value
            rfp_filename = selected_path.name
            
            print(f"📄 Loading: {rfp_filename}")
            print("🔄 Extracting text...")
            
            text_parts = []
            
            try:
                with pdfplumber.open(selected_path) as pdf:
                    for page_num, page in enumerate(pdf.pages, 1):
                        page_text = page.extract_text()
                        if page_text:
                            text_parts.append(f"\n--- Page {page_num} ---\n{page_text}")
                
                rfp_text = "\n".join(text_parts)
                
                print(f"✅ Successfully extracted {len(rfp_text)} characters from {len(text_parts)} pages")
                print(f"\n📝 Preview (first 500 chars):\n")
                print(rfp_text[:500] + "...")
                
            except Exception as e:
                print(f"❌ Error loading RFP: {e}")
                rfp_text = None
    
    load_button.on_click(on_load_click)
    
    display(VBox([rfp_dropdown, load_button, output]))

### ✍️ Option C: Paste RFP text directly

In [ ]:
# Uncomment and paste your RFP text here if you want to use this option

# rfp_text = """
# Paste your RFP text here...
# """
# 
# rfp_filename = "Manual_RFP"
# print(f"✅ RFP text set manually ({len(rfp_text)} characters)")

---

## 4️⃣ Generate Proposal Approach Automatically

This cell runs the complete analysis pipeline:
1. Extract workstreams from RFP
2. Learn from repository
3. Analyze dependencies
4. Build detailed modules
5. Optimize timeline

**⏱️ This takes 2-5 minutes depending on RFP complexity**

In [ ]:
if rfp_text is None:
    print("❌ No RFP loaded! Please use one of the options above to load an RFP first.")
else:
    print("🚀 Starting automated analysis...\n")
    print("=" * 60)
    
    # Initialize OpenAI settings
    openai_api_key = os.getenv('OPENAI_API_KEY')
    openai_model = os.getenv('OPENAI_MODEL', 'gpt-4')
    openai_temperature = float(os.getenv('OPENAI_TEMPERATURE', 0.1))
    
    print(f"🤖 Using model: {openai_model}")
    print("=" * 60 + "\n")
    
    # Step 1: Extract workstreams
    print("📋 STEP 1: Extracting workstreams from RFP...\n")
    
    ws_agent = WorkstreamAgent(
        api_key=openai_api_key,
        model=openai_model,
        temperature=openai_temperature,
        verbose=True
    )
    
    workstream_names, duration, reasoning = ws_agent.extract_explicit_workstreams(rfp_text)
    
    print(f"\n✅ Found {len(workstream_names)} workstreams")
    for idx, ws_name in enumerate(workstream_names, 1):
        print(f"   {idx}. {ws_name}")
    
    if duration:
        print(f"\n📅 Estimated duration: {duration} weeks")
    
    # Step 2: Learn from repository
    print("\n" + "=" * 60)
    print("🧠 STEP 2: Learning from repository...\n")
    
    enhanced_workstreams, learning_reasoning = ws_agent.learn_from_repository(rfp_text, rag_engine)
    
    print(f"\n✅ Generated {len(enhanced_workstreams)} enhanced workstreams")
    for ws in enhanced_workstreams:
        print(f"   - {ws.name}: {ws.duration_weeks} weeks")
    
    # Step 3: Analyze dependencies
    print("\n" + "=" * 60)
    print("🔗 STEP 3: Analyzing dependencies...\n")
    
    dep_agent = DependencyAgent(
        api_key=openai_api_key,
        model=openai_model,
        temperature=openai_temperature,
        verbose=True
    )
    
    dependencies, dep_reasoning = dep_agent.analyze_dependencies(
        workstreams=enhanced_workstreams,
        rag_engine=rag_engine,
        rfp_context=rfp_text[:2000]
    )
    
    print(f"\n✅ Dependency analysis complete")
    print(f"   - Parallel opportunities identified: {len(dependencies.parallel_workstreams)}")
    
    # Step 4: Build modules
    print("\n" + "=" * 60)
    print("🏗️ STEP 4: Building detailed modules...\n")
    
    module_agent = ModuleAgent(
        api_key=openai_api_key,
        model=openai_model,
        temperature=openai_temperature,
        verbose=True
    )
    
    all_module_names = set()
    total_modules = 0
    
    for workstream in enhanced_workstreams:
        print(f"\n🔧 Building modules for: {workstream.name}")
        
        modules, mod_reasoning = module_agent.build_modules(
            workstream=workstream,
            all_workstreams=enhanced_workstreams,
            rag_engine=rag_engine,
            project_context=rfp_text[:2000]
        )
        
        workstream.modules = modules
        total_modules += len(modules)
        
        for module in modules:
            all_module_names.add(module.name)
        
        print(f"   ✅ Created {len(modules)} modules")
    
    print(f"\n✅ Module building complete")
    print(f"   - Total modules: {total_modules}")
    print(f"   - Unique module names: {len(all_module_names)} (MECE validated)")
    
    # Step 5: Optimize timeline
    print("\n" + "=" * 60)
    print("⚡ STEP 5: Optimizing timeline...\n")
    
    optimizer = TimelineOptimizer(
        api_key=openai_api_key,
        model=openai_model,
        temperature=openai_temperature,
        verbose=True
    )
    
    original_duration = max(ws.start_week + ws.duration_weeks for ws in enhanced_workstreams)
    
    optimization_result, optimized_workstreams, opt_reasoning = optimizer.optimize(
        workstreams=enhanced_workstreams,
        dependencies=dependencies,
        rag_engine=rag_engine,
        constraints={'max_parallel_workstreams': 3}
    )
    
    optimized_duration = max(ws.start_week + ws.duration_weeks for ws in optimized_workstreams)
    
    print(f"\n✅ Timeline optimization complete")
    print(f"   - Original duration: {original_duration} weeks")
    print(f"   - Optimized duration: {optimized_duration} weeks")
    print(f"   - Weeks saved: {optimization_result.weeks_saved}")
    print(f"   - Reduction: {optimization_result.percent_reduction:.1f}%")
    
    # Create project structure
    print("\n" + "=" * 60)
    print("💾 Creating project structure...\n")
    
    project_name = rfp_filename.replace('.pdf', '') if rfp_filename else "New_RFP_Analysis"
    
    project = ProjectStructure(
        project_name=project_name,
        rfp_text=rfp_text,
        workstreams=optimized_workstreams,
        dependencies=dependencies,
        agent_reasoning_log=[reasoning, learning_reasoning, dep_reasoning, opt_reasoning]
    )
    
    # Save project
    saved_path = save_project(project)
    
    print(f"✅ Project saved: {saved_path}")
    
    print("\n" + "=" * 60)
    print("🎉 ANALYSIS COMPLETE!")
    print("=" * 60)
    
    print(f"\n📊 Summary:")
    print(f"   - Project: {project_name}")
    print(f"   - Workstreams: {len(project.workstreams)}")
    print(f"   - Total Modules: {sum(len(ws.modules) for ws in project.workstreams)}")
    print(f"   - Total Activities: {sum(sum(len(m.activities) for m in ws.modules) for ws in project.workstreams)}")
    print(f"   - Timeline: {optimized_duration} weeks")
    print(f"   - Optimization: {optimization_result.weeks_saved} weeks saved")
    
    print("\n📥 Next: Run the cells below to export your proposal approach!")

---

## 5️⃣ Export Proposal Approach

Generate professional Excel and PowerPoint files with your proposal structure.

In [ ]:
if 'project' not in locals():
    print("❌ No project generated yet. Please run the analysis cell above first.")
else:
    print("📤 Exporting to Excel and PowerPoint...\n")
    
    exported_files = export_project(
        project,
        export_excel=True,
        export_pptx=True
    )
    
    print("\n✅ Export complete!\n")
    print("📁 Files created:")
    for format_name, filepath in exported_files.items():
        print(f"   - {format_name}: {filepath}")
    
    print("\n💡 These files contain your complete proposal approach section!")
    print("   Open them to review the workstreams, modules, activities, and timeline.")

---

## 6️⃣ Generate Interactive Visualizations

Create visual timeline and charts for your proposal.

In [ ]:
if 'project' not in locals():
    print("❌ No project generated yet. Please run the analysis cell above first.")
else:
    print("🎨 Creating visualizations...\n")
    
    viz = ProjectVisualizer(project)
    
    # Gantt chart
    print("📊 Generating Gantt chart...")
    fig = viz.create_gantt_chart(title=f"{project.project_name} - Timeline")
    fig.show()
    
    print("\n💡 Hover over bars to see details. Use toolbar to zoom and save image.")

In [ ]:
if 'viz' in locals():
    # Dashboard
    print("📊 Generating comprehensive dashboard...")
    fig = viz.create_summary_dashboard(optimization_result)
    fig.show()

In [ ]:
if 'viz' in locals():
    # Export all visualizations
    print("💾 Exporting all visualizations to HTML...\n")
    
    exported_viz = viz.export_all_charts()
    
    print("\n✅ Visualizations exported!\n")
    print("📁 Files created:")
    for name, path in exported_viz.items():
        print(f"   - {name}: {path}")
    
    print("\n💡 Open these HTML files in your browser to view interactive charts.")

---

## 🎉 You're Done!

Your proposal approach section has been generated and exported.

### 📂 Check these folders for your outputs:

1. **`output/projects/`** - Saved project file (can be loaded later)
2. **`output/exports/`** - Excel and PowerPoint files
3. **`output/visualizations/`** - Interactive HTML charts

### 🔄 To Analyze Another RFP:

1. Go back to **Section 3** and upload/select a different RFP
2. Run **Section 4** to generate the approach
3. Run **Sections 5-6** to export

### 💡 For More Control:

Use the detailed notebook `01_Enterprise_RAG_Architecture.ipynb` which allows you to:
- Review and edit at each stage
- Manually refine workstreams, modules, and activities
- See full AI reasoning and evidence

---

*Version 2.0.0*  
*Quick Document Analyzer - Automated proposal approach generation*